# Create SEPM Science Awards

Creates OpenAlex award rows from Society for Sedimentary Geology (SEPM) official science-award recipient pages.

**Awarding body:** Society for Sedimentary Geology - F4320312534 (US; no ROR or DOI in OpenAlex)

**Sources:**
- https://www.sepm.org/Past-Winners
- https://www.sepm.org/2024awardees
- https://www.sepm.org/2026awardees

**Source method:** official static HTML pages. The past-winners page publishes historical science award and honor recipients through 2023. The 2024 and 2026 awardee pages add current cohorts not yet folded into the past-winners page.

**Local validation on 2026-05-28:** 369 rows, 369 unique native IDs, years 1930-2026, 100% recipient/name split/award/year/landing-page coverage, 7 current-page affiliation rows, and amount/currency NULL by prize-pattern waiver.

**S3 staging path:** `s3a://openalex-ingest/awards/sepm/sepm_science_awards.parquet`

**Schema choices:**
- One row per (award, year, recipient).
- `funder_award_id` is a deterministic key from year, award name, recipient name, and a short source-field hash.
- `funding_type` is `prize` for all rows.
- `funder_scheme` is the SEPM award/honor name.
- `lead_investigator` is the individual recipient. Current pages include affiliation/email for seven 2024 recipients; historical rows do not publish affiliations, so those affiliation fields remain NULL.
- `amount` and `currency` remain NULL under the prize-pattern waiver because SEPM does not publish per-recipient monetary amounts for these honors.


## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sepm_science_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/sepm/sepm_science_awards.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 369 rows.
SELECT COUNT(*) AS total_sepm_awards
FROM openalex.awards.sepm_science_awards_raw;


In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.sepm_science_awards_raw;


In [ ]:
%sql
-- Sample raw SEPM data.
SELECT
    funder_award_id,
    display_name,
    recipient_name,
    recipient_given_name,
    recipient_family_name,
    award_name,
    award_year,
    affiliation_name,
    affiliation_country,
    landing_page_url
FROM openalex.awards.sepm_science_awards_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. Do not wrap DESCRIBE in a subquery.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'sepm_science_awards_raw'
  AND LOWER(column_name) RLIKE 'amount|amt|total|funding|award|grant|currency|usd|dollar|value|budget|cost'
ORDER BY ordinal_position;


In [ ]:
%sql
-- Confirm prize-pattern amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    ROUND(try_divide(COUNT(currency), COUNT(*)) * 100.0, 1) AS pct_with_currency,
    COUNT(award_year) AS rows_with_award_year,
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year
FROM openalex.awards.sepm_science_awards_raw;


In [ ]:
%sql
-- Native-key inspection.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.sepm_science_awards_raw;


In [ ]:
%sql
-- Award and source distribution.
SELECT
    award_name,
    source_record_type,
    COUNT(*) AS rows
FROM openalex.awards.sepm_science_awards_raw
GROUP BY award_name, source_record_type
ORDER BY rows DESC;


In [ ]:
%sql
-- Year distribution.
SELECT award_year, COUNT(*) AS rows
FROM openalex.awards.sepm_science_awards_raw
GROUP BY award_year
ORDER BY TRY_CAST(award_year AS INT);


In [ ]:
%sql
-- Recipient and affiliation coverage.
SELECT
    COUNT(*) AS total,
    COUNT(recipient_given_name) AS has_given_name,
    COUNT(recipient_family_name) AS has_family_name,
    COUNT(affiliation_name) AS has_affiliation,
    ROUND(try_divide(COUNT(affiliation_name), COUNT(*)) * 100.0, 1) AS pct_affiliation,
    COUNT(affiliation_country) AS has_affiliation_country
FROM openalex.awards.sepm_science_awards_raw;


## Step 1.6: Funder Existence Fail-Fast

Runbook Step 2.2.4 mandatory check. Must return exactly 1 row for Society for Sedimentary Geology.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320312534;


In [ ]:
%sql
-- Fail-fast count form. Expected: exactly one row.
SELECT COUNT(*) AS funder_rows
FROM openalex.common.funder
WHERE funder_id = 4320312534;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.sepm_science_awards
USING delta
AS
WITH
sepm_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320312534
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(award_year AS INT) AS parsed_year,
        CASE
            WHEN TRY_CAST(award_year AS INT) BETWEEN 1900 AND YEAR(current_date()) + 1
            THEN TRY_TO_DATE(CONCAT(award_year, '-01-01'), 'yyyy-MM-dd')
            ELSE NULL
        END AS parsed_start_date
    FROM openalex.awards.sepm_science_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 AS id,

        TRIM(g.display_name) AS display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END AS description,

        f.funder_id,
        g.native_award_id AS funder_award_id,

        CAST(NULL AS DOUBLE) AS amount,
        CAST(NULL AS STRING) AS currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) AS id,
            f.display_name,
            f.ror_id,
            f.doi
        ) AS funder,

        'prize' AS funding_type,

        NULLIF(TRIM(g.award_name), '') AS funder_scheme,

        'sepm_science_awards' AS provenance,

        g.parsed_start_date AS start_date,
        CAST(NULL AS DATE) AS end_date,
        g.parsed_year AS start_year,
        CAST(NULL AS INT) AS end_year,

        struct(
            NULLIF(TRIM(g.recipient_given_name), '') AS given_name,
            NULLIF(TRIM(g.recipient_family_name), '') AS family_name,
            CAST(NULL AS STRING) AS orcid,
            g.parsed_start_date AS role_start,
            struct(
                NULLIF(TRIM(g.affiliation_name), '') AS name,
                NULLIF(TRIM(g.affiliation_country), '') AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        ) AS lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) AS co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) AS investigators,

        NULLIF(TRIM(g.landing_page_url), '') AS landing_page_url,
        CAST(NULL AS STRING) AS doi,

        CONCAT('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) AS works_api_url,

        current_timestamp() AS created_date,
        current_timestamp() AS updated_date

    FROM raw_prepared g
    CROSS JOIN sepm_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Delete Previous Source Rows and Insert Priority 170

In [ ]:
%sql
-- Remove previous SEPM science award data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sepm_science_awards' AND priority = 170;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    170 AS priority
FROM openalex.awards.sepm_science_awards;


## Step 6: Verification

In [ ]:
%sql
SELECT COUNT(*) AS total_sepm_science_awards
FROM openalex.awards.sepm_science_awards;


In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.sepm_science_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_year,
    lead_investigator.given_name AS recipient_given_name,
    lead_investigator.family_name AS recipient_family_name,
    lead_investigator.affiliation.name AS affiliation_name,
    landing_page_url
FROM openalex.awards.sepm_science_awards
LIMIT 10;


In [ ]:
%sql
-- Check OpenAlex ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids
FROM openalex.awards.sepm_science_awards;


In [ ]:
%sql
-- Check funder distribution. Should be only Society for Sedimentary Geology.
SELECT funder.display_name, funder_id, COUNT(*) AS rows
FROM openalex.awards.sepm_science_awards
GROUP BY funder.display_name, funder_id
ORDER BY rows DESC;


In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    COUNT(start_year) AS has_start_year,
    COUNT(landing_page_url) AS has_url,
    COUNT(lead_investigator) AS has_recipient,
    COUNT(lead_investigator.given_name) AS has_given_name,
    COUNT(lead_investigator.family_name) AS has_family_name,
    COUNT(lead_investigator.affiliation.name) AS has_affiliation,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) AS pct_affiliation
FROM openalex.awards.sepm_science_awards;


In [ ]:
%sql
-- Prize-pattern amount waiver check. Expected: zero amount/currency rows.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    COUNT(currency) AS has_currency,
    collect_set(currency) AS currencies
FROM openalex.awards.sepm_science_awards;


In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) AS rows
FROM openalex.awards.sepm_science_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year;


In [ ]:
%sql
-- Award distribution.
SELECT funding_type, funder_scheme, COUNT(*) AS rows
FROM openalex.awards.sepm_science_awards
GROUP BY funding_type, funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 170.
SELECT
    priority,
    provenance,
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'sepm_science_awards' AND priority = 170
GROUP BY priority, provenance;


## Handoff Notes

Contractor has no S3 or Databricks access. Admin should:

1. Run `scripts/local/sepm_science_awards_to_s3.py --skip-download` without `--skip-upload` after reviewing the local parquet.
2. Run this notebook on Databricks.
3. Inspect Step 6 verification cells for row count, uniqueness, funder mapping, amount-waiver behavior, and priority 170 insertion.
4. Only add scheduled job YAML after upload, notebook execution, and QA pass.
